# Day 4 — Streamlit & System Architecture

## 1. Streamlit 소개: Python만으로 만드는 웹 UI

### Streamlit 패키지 확인

In [2]:
# !pip list | grep stream
!pip list | findstr streamlit    

streamlit                 1.38.0



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
os.makedirs("frontend", exist_ok=True)

try:
    import streamlit
    print(f"✅ Streamlit: {streamlit.__version__}")
except ImportError:
    print("❌ Streamlit 미설치")
    print("  pip install streamlit")

✅ Streamlit: 1.38.0


### 1.4 실습: 첫 번째 Streamlit 앱

In [5]:
%%writefile frontend/app_hello.py
"""
Day 4 - 첫 번째 Streamlit 앱
"""
import streamlit as st

# 페이지 설정
st.set_page_config(
    page_title="My First Streamlit App",
    page_icon="🤖",
    layout="centered",
)

# 제목
st.title("🤖 나의 첫 Streamlit 앱")
st.write("Python 코드만으로 이 화면이 만들어졌습니다.")

# 구분선
st.divider()

# 텍스트 입력
name = st.text_input("이름을 입력하세요:", placeholder="홍길동")

# 조건부 출력
if name:
    st.success(f"안녕하세요, {name}님! 환영합니다. 🎉")
else:
    st.info("위에 이름을 입력해 보세요.")

# 버튼
if st.button("날짜 확인"):
    from datetime import datetime
    now = datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")
    st.write(f"현재 시각: {now}")

Writing frontend/app_hello.py


### 실행 방법

```
# 터미널에서 실행 (별도 터미널을 열거나, 노트북에서 !로 실행)
# 주의: !로 실행하면 노트북 셀이 블로킹되므로, 별도 터미널 권장

# 로컬 환경:
streamlit run frontend/app_hello.py --server.port 8501

# Colab 환경:
# !streamlit run frontend/app_hello.py --server.port 8501 &
# (백그라운드 실행 후 ngrok 또는 localtunnel로 외부 접속)
```

In [6]:
# 노트북에서 백그라운드로 실행하는 방법 (선택)
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_hello.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("✅ Streamlit 앱 실행 중: http://localhost:8501")
print("   브라우저에서 위 주소로 접속하세요.")

# Colab:
# from google.colab import output
# output.serve_kernel_port_as_iframe(8501)

✅ Streamlit 앱 실행 중: http://localhost:8501
   브라우저에서 위 주소로 접속하세요.


## 4. Streamlit에서 FastAPI 호출하기

### 4.1 기본 패턴

In [7]:
import requests

# GET — 서버 상태 확인
response = requests.get("http://localhost:8000/health")
health = response.json()

# POST — 추론 요청 (Base64 이미지)
import base64

image_base64 = base64.b64encode(image_bytes).decode("utf-8")
response = requests.post(
    "http://localhost:8000/predict/image",
    json={"image_base64": image_base64, "return_probabilities": True},
)
result = response.json()

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다"))

### 4.2 에러 처리

In [8]:
import streamlit as st
import requests

def call_api(url, json_data=None, method="post"):
    """API를 호출하고, 실패 시 st.error()로 안내합니다."""
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버가 실행 중인지 확인하세요.")
        return None
    except requests.exceptions.Timeout:
        st.warning("⏱️ **응답 시간 초과.** 잠시 후 다시 시도하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None

## 5. 실습: MNIST 추론 대시보드 만들기

### 5.2 대시보드 코드 작성

In [10]:
%%writefile frontend/app_dashboard.py
"""
Day 4 - MNIST 추론 대시보드
FastAPI 백엔드와 연동하는 Streamlit 프론트엔드
"""
import streamlit as st
import requests
import base64
import io
from PIL import Image


# ===== 페이지 설정 =====
st.set_page_config(
    page_title="MNIST 숫자 인식",
    page_icon="🔢",
    layout="wide",
)

# debug 로그 박스
if "debug_logs" not in st.session_state:
    st.session_state.debug_logs = []

def log_debug(msg: str):
    st.session_state.debug_logs.append(msg)
    
# 초기화 함수
def reset_app():
    st.session_state["image_bytes"] = None
    st.session_state.pop("last_result", None)
    st.session_state.pop("sample_label", None)
    st.session_state["debug_logs"] = []


# ===== API 호출 함수 =====
API_BASE = "http://localhost:8000"

def call_api(url, json_data=None, method="post"):
    """API를 호출하고, 실패 시 에러 메시지를 표시합니다."""
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)

        log_debug(f"DEBUG status_code: {resp.status_code}")
        if resp.status_code not in [200, 201]:
            log_debug(f"DEBUG raw text: {resp.text}")
        
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버가 실행 중인지 확인하세요.")
        return None
    except requests.exceptions.Timeout:
        st.warning("⏱️ **응답 시간 초과.** 잠시 후 다시 시도하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None

# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")

    # 서버 상태 표시
    health = call_api(f"{API_BASE}/health", method="get")
    if health and health.get("status") == "healthy":
        st.success("🟢 서버 연결됨")
        server_ok = True
    else:
        st.error("🔴 서버 연결 실패")
        server_ok = False

    st.divider()

    # 옵션
    show_probabilities = st.checkbox("전체 확률 표시", value=True)
    show_preprocessed = st.checkbox("전처리된 이미지 표시", value=True)

    st.divider()
    st.caption("MNIST Prediction Dashboard v1.0")

    # 초기화
    st.divider()
    if st.button("🔄 초기화", use_container_width=True):
        reset_app()
        st.rerun()  # 즉시 새상태로 다시 실행

    # debug log box
    st.divider()
    st.subheader("debug log")

    if st.session_state.debug_logs:
        st.info("\n".join(st.session_state.debug_logs))
    else:
        st.caption("아직 로그가 없습니다.")
    


# ===== 메인 영역 =====
st.title("🔢 MNIST 숫자 인식")
st.write("손글씨 숫자 이미지를 업로드하면 0~9 중 어떤 숫자인지 예측합니다.")

col_input, col_result = st.columns(2)


# ----- 입력 영역 -----
with col_input:
    st.subheader("📤 이미지 입력")

    input_method = st.radio(
        "입력 방식:", ["파일 업로드", "샘플 이미지 사용"], horizontal=True,
    )

    image_bytes = None
    if "image_bytes" not in st.session_state:
        st.session_state["image_bytes"] = image_bytes

    if input_method == "파일 업로드":
        uploaded = st.file_uploader(
            "이미지를 업로드하세요:",
            type=["png", "jpg", "jpeg"],
            help="28x28 그레이스케일 권장. 다른 크기도 자동 변환됩니다.",
        )
        if uploaded:
            image_bytes = uploaded.getvalue()
            st.session_state["image_bytes"] = image_bytes
            st.image(uploaded, caption="업로드된 이미지", width=200)

    else:
        st.info("샘플 이미지를 사용하려면 아래 버튼을 누르세요.")
        sample_idx = st.number_input("샘플 번호 (0~99):", min_value=0, max_value=99, value=0)

        if st.button("샘플 이미지 로드"):
            try:
                from torchvision import datasets
                test_dataset = datasets.MNIST(root="data", train=False, download=True)
                sample_image, sample_label = test_dataset[sample_idx]

                buffer = io.BytesIO()
                sample_image.save(buffer, format="PNG")
                image_bytes = buffer.getvalue()
                st.session_state["image_bytes"] = image_bytes

                st.image(sample_image, caption=f"샘플 #{sample_idx} (정답: {sample_label})", width=200)
                st.session_state["sample_label"] = sample_label
            except Exception as e:
                st.error(f"샘플 로드 실패: {e}")

    # 전처리된 이미지 미리보기
    if image_bytes and show_preprocessed:
        st.caption("전처리된 이미지 (28x28 그레이스케일):")
        img = Image.open(io.BytesIO(image_bytes)).convert("L").resize((28, 28))
        st.image(img, width=150)


# ----- 결과 영역 -----
with col_result:
    st.subheader("📊 추론 결과")

    if "image_bytes" in st.session_state:
        image_bytes = st.session_state["image_bytes"]

    if image_bytes is None:
        st.info("👈 왼쪽에서 이미지를 업로드하거나 샘플을 선택하세요.")

    elif not server_ok:
        st.error("서버에 연결할 수 없습니다. 사이드바의 서버 상태를 확인하세요.")

    else:
        run = st.button("🚀 추론 실행", type="primary", use_container_width=True)

        if run:
            with st.spinner("모델 추론 중..."):
                # Base64 인코딩 → API 호출
                image_base64 = base64.b64encode(image_bytes).decode("utf-8")

                result = call_api(
                    f"{API_BASE}/predict/image",
                    json_data={
                        "image_base64": image_base64,
                        "return_probabilities": show_probabilities,
                    },
                )
                
            if result is not None:
                st.session_state["last_result"] = result

        # 결과 표시
        if "last_result" in st.session_state:
            result = st.session_state["last_result"]

            # 메트릭
            m1, m2 = st.columns(2)
            with m1:
                st.metric(label="예측 결과", value=result["predicted_class"])
            with m2:
                st.metric(label="확신도", value=f"{result['confidence']:.1%}")

            # 확률 분포
            if result.get("probabilities"):
                st.subheader("📊 클래스별 확률 분포")
                probs = result["probabilities"]
                for cls in sorted(probs.keys(), key=lambda x: int(x)):
                    prob = probs[cls]
                    c1, c2 = st.columns([1, 5])
                    with c1:
                        is_pred = cls == result["predicted_class"]
                        st.write(f"**{'👉 ' if is_pred else ''}{cls}**")
                    with c2:
                        st.progress(float(prob), text=f"{prob:.2%}")

            # 샘플 이미지인 경우 정답 비교
            if "sample_label" in st.session_state:
                label = st.session_state["sample_label"]
                if result["predicted_class"] == str(label):
                    st.success(f"✅ 정답! (정답: {label})")
                else:
                    st.error(f"❌ 오답 (정답: {label}, 예측: {result['predicted_class']})")

Overwriting frontend/app_dashboard.py


### 5.3 실행 및 테스트

#### Step 1: 백엔드 서버 실행

In [1]:
# ⚠️ 이전 Day/섹션에서 서버를 실행했다면 커널을 재시작하세요.
#    "Address already in use" → Kernel → Restart Kernel

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_backend():
    uvicorn.run("app.main_final:app", host="0.0.0.0", port=8000)

backend_thread = threading.Thread(target=run_backend, daemon=True)
backend_thread.start()
time.sleep(3)
print("✅ 백엔드 서버: http://localhost:8000")

# terminal 직접 실행 방법
# uvicorn project.app.main_final:app --host 0.0.0.0 --port 8000 --reload

✅ 백엔드 서버: http://localhost:8000


INFO:     Started server process [4172]
INFO:     Waiting for application startup.


2026-04-01 23:46:38 INFO     [ml_api] 모델 로드 중: models/mnist_state_dict.pth


INFO:ml_api:모델 로드 중: models/mnist_state_dict.pth


2026-04-01 23:46:38 INFO     [ml_api] 모델 로드 완료


INFO:ml_api:모델 로드 완료
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


#### Step 2: 프론트엔드 실행

###### Window 에서 서버 또는 streamlit 화면 kill

- 1. 기 실행한 streamlit 이 포트 8501 사용한다고 나올때
```
1. Ctrl + c로 현재 터미널 종로 -> 새 터미널에서 재실행

2. 포트 점유 프로세스 종료

netstat -ano | findstr :8501
taskkill /PID 23352 /F
```

- 2. 서버 kill
```
tasklist | findstr python
taskkill /PID 5428 /F
```

In [11]:
# 별도 터미널에서 실행합니다
# cd model-serving-course
# streamlit run project/frontend/app_dashboard.py --server.port 8501

2026-04-01 12:24:00 INFO     [ml_api] GET /health -> 200 (0.0s)


INFO:ml_api:GET /health -> 200 (0.0s)


INFO:     127.0.0.1:11066 - "GET /health HTTP/1.1" 200 OK
2026-04-01 12:24:11 INFO     [ml_api] GET /health -> 200 (0.0s)


INFO:ml_api:GET /health -> 200 (0.0s)


INFO:     127.0.0.1:2521 - "GET /health HTTP/1.1" 200 OK
2026-04-01 12:24:35 INFO     [ml_api] GET /health -> 200 (0.0s)


...

INFO:     127.0.0.1:10026 - "GET /health HTTP/1.1" 200 OK
2026-04-01 12:27:55 INFO     [ml_api] GET /health -> 200 (0.0s)


INFO:ml_api:GET /health -> 200 (0.0s)


INFO:     127.0.0.1:10027 - "GET /health HTTP/1.1" 200 OK
2026-04-01 12:27:59 INFO     [ml_api] GET /health -> 200 (0.001s)


INFO:ml_api:GET /health -> 200 (0.001s)


INFO:     127.0.0.1:10031 - "GET /health HTTP/1.1" 200 OK


In [ ]:
# 또는 노트북에서 백그라운드 실행
import subprocess
proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_dashboard.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("✅ 프론트엔드: http://localhost:8501")

# Colab: from google.colab import output
# output.serve_kernel_port_as_iframe(8501)

### Step 3: 테스트 시나리오

```
브라우저에서 http://localhost:8501 에 접속합니다.

[테스트 1] 기본 흐름
  1. 사이드바에서 서버 상태가 🟢인지 확인
  2. "샘플 이미지 사용" → 샘플 번호 0 → "샘플 이미지 로드"
  3. "🚀 추론 실행" 클릭
  4. 예측 숫자, 확신도, 확률 분포가 표시되는지 확인

[테스트 2] 파일 업로드
  1. "파일 업로드" 선택 → 아무 이미지 드래그 앤 드롭
  2. "🚀 추론 실행" → 결과 확인

[테스트 3] 에러 상황
  1. 백엔드 서버를 종료
  2. 사이드바 상태가 🔴으로 변하는지 확인
```